# Laboratory: Neural Models — Learning, Depth, Activations, and Output Layers

**Course Laboratory Exercise**  
**Focus Areas:** Theoretical Foundations of Deep Learning, Mathematical Derivations, PyTorch Reverse-Mode AD, Optimization Diagnostics, and Engineering vs. Scientific Analysis.

---

## 1. Executive Summary & Conceptual Foundations of Neural Models

A **neural model** is a parameterized function $f_\theta: \mathcal{X} \to \mathcal{Y}$ mapping inputs $x \in \mathcal{X}$ to outputs or predictive distributions over $\mathcal{Y}$.

### Layer Decomposition & The Affine Collapse Problem
For a layered feed-forward neural network of depth $L$, computation proceeds recursively:
$$
a^{(\ell)} = W^{(\ell)} h^{(\ell-1)} + b^{(\ell)}, \quad h^{(\ell)} = f^{(\ell)}(a^{(\ell)}), \quad \text{with } h^{(0)} = x
$$
where:
- $a^{(\ell)} \in \mathbb{R}^{d_\ell}$ is the pre-activation vector at layer $\ell$.
- $W^{(\ell)} \in \mathbb{R}^{d_\ell \times d_{\ell-1}}$ and $b^{(\ell)} \in \mathbb{R}^{d_\ell}$ are the weight matrix and bias vector.
- $f^{(\ell)}: \mathbb{R} \to \mathbb{R}$ is an element-wise activation function.
- $h^{(\ell)}$ is the hidden representation (or feature vector).

**Why Nonlinearity is Essential (The Affine Collapse Theorem):**  
If every layer is purely affine ($f^{(\ell)}(z) = z$), then a two-layer composition collapses into a single affine transformation:
$$
h^{(2)} = W^{(2)}(W^{(1)} x + b^{(1)}) + b^{(2)} = (W^{(2)} W^{(1)}) x + (W^{(2)} b^{(1)} + b^{(2)}) = \tilde{W} x + \tilde{b}
$$
Thus, depth without nonlinearity yields no added expressive power beyond a single linear hyperplane. Nonlinear activations fold, warp, and partition input space, allowing non-convex decision regions to be separated.

### Output Layers, Loss Functions & Information Geometry
- **Binary Classification:** Outputs represent Bernoulli parameter $p = P(y=1|x) \in [0, 1]$. We pair a raw logit $z \in \mathbb{R}$ with the **sigmoid** activation $\sigma(z) = \frac{1}{1 + e^{-z}}$ and **Binary Cross-Entropy (BCE)** loss.
- **Multiclass Classification ($K$ classes):** Outputs represent a Categorical distribution on the simplex $\Delta^{K-1}$. We pair logits $z \in \mathbb{R}^K$ with the **softmax** activation $p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$ and **Categorical Cross-Entropy** loss. Crucially, the derivative of cross-entropy with respect to the pre-softmax logits simplifies cleanly to $\nabla_z \mathcal{L} = p - y$.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# Ensure deterministic reproducibility
torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch Version: {torch.__version__}")

## Task 1: Understand the Problem Before Coding

### 1. Problem Specification
- **Input Space:** $\mathcal{X} = \{0, 1\}^2 \subset \mathbb{R}^2$ representing redundant safety sensor states $(x_1, x_2)$.
- **Output Space:** $\mathcal{Y} = \{0, 1\}$ where $y=1$ denotes a sensor-disagreement warning.
- **Labelled Dataset:**
  1. $x^{(1)} = (0, 0) \implies y^{(1)} = 0$ (Both sensors inactive)
  2. $x^{(2)} = (0, 1) \implies y^{(2)} = 1$ (Sensor 2 active, disagreement)
  3. $x^{(3)} = (1, 0) \implies y^{(3)} = 1$ (Sensor 1 active, disagreement)
  4. $x^{(4)} = (1, 1) \implies y^{(4)} = 0$ (Both sensors active)

### 2. Linear Non-Separability Explanation
A linear decision boundary in $\mathbb{R}^2$ is a line $w_1 x_1 + w_2 x_2 + b = 0$. For complete separation, points $(0,1)$ and $(1,0)$ must lie on one side (e.g. $>0$), while $(0,0)$ and $(1,1)$ must lie on the other side ($<0$). This requires:
$$
b < 0, \quad w_1 + b > 0, \quad w_2 + b > 0, \quad w_1 + w_2 + b < 0
$$
Summing the middle two inequalities yields $(w_1 + w_2 + 2b) > 0$. However, the first and fourth imply $(w_1 + w_2 + b) + b < 0$, which is a direct mathematical contradiction. Geometrically, the line connecting $(0,1)$ and $(1,0)$ intersects the line connecting $(0,0)$ and $(1,1)$ at $(0.5, 0.5)$. Since the convex hulls of the two classes intersect, they cannot be separated by any single hyperplane.

### 3. Prediction for Linear Model
A single affine layer followed by a sigmoid will fail to separate the four points. It can at best separate 3 out of 4 points (75% accuracy), or predict $p \approx 0.5$ across all inputs, leading to a stalled cross-entropy loss of $\ln(2) \approx 0.6931$.

In [ ]:
# Dataset definition
X = torch.tensor([[0.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 0.0],
                  [1.0, 1.0]], dtype=torch.float32)
y_xor = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

# Train a purely linear model (Affine + Sigmoid)
torch.manual_seed(42)
linear_model = nn.Linear(2, 1)
optimizer_lin = optim.SGD(linear_model.parameters(), lr=0.1)
criterion_lin = nn.BCEWithLogitsLoss()

for step in range(2000):
    optimizer_lin.zero_grad()
    out = linear_model(X)
    loss = criterion_lin(out, y_xor)
    loss.backward()
    optimizer_lin.step()

with torch.no_grad():
    probs_lin = torch.sigmoid(linear_model(X))
    preds_lin = (probs_lin > 0.5).int()

print(f"Linear Model Final Loss: {loss.item():.4f} (Stalled near ln(2) = 0.6931)")
print(f"Linear Model Predictions: {preds_lin.squeeze().tolist()} | Ground Truth: {y_xor.squeeze().int().tolist()}")
print(f"Accuracy: {(preds_lin == y_xor).float().mean().item()*100:.1f}%")

### Think About It (Task 1)
> **Prompt:** *A model can have many parameters and still have the wrong kind of representation. What scientific claim about representation does XOR let us test with only four data points?*

**Answer:** XOR proves the fundamental scientific claim that **capacity is not merely a function of parameter count, but of representational geometry**. Even if an affine network is made arbitrarily wide or deep ($k$ layers with millions of weights), it can only represent affine transformations $W x + b$. Because XOR requires non-convex partitioning, no purely linear hypothesis class can represent it, regardless of parameter count. XOR is the minimal four-point counterexample proving that **nonlinear feature coordinate transformations are strictly required** for nonlinear decision problems.

## Task 2: Design the Intelligent Agent

### Baseline Architecture Specification
$$
\begin{aligned}
h^{(1)} &= f\left(W^{(1)} x + b^{(1)}\right), \quad W^{(1)} \in \mathbb{R}^{2 \times 2}, \, b^{(1)} \in \mathbb{R}^2 \\
z &= W^{(2)} h^{(1)} + b^{(2)}, \quad W^{(2)} \in \mathbb{R}^{1 \times 2}, \, b^{(2)} \in \mathbb{R} \\
\hat{y} &= \sigma(z) = \frac{1}{1 + e^{-z}}
\end{aligned}
$$

### Task 2 Questions & Answers
1. **Why is the hidden nonlinearity scientifically necessary here?**  
   Without the activation function $f$, $z = W^{(2)}(W^{(1)}x + b^{(1)}) + b^{(2)} = (W^{(2)}W^{(1)})x + (W^{(2)}b^{(1)} + b^{(2)})$, which collapses to a single hyperplane. The hidden units must map the input space $(x_1, x_2)$ nonlinearly into a latent feature space $(h_1, h_2)$ where the two classes become linearly separable (e.g. mapping $(0,0)$ and $(1,1)$ to low values on one feature axis and $(0,1)$ and $(1,0)$ to high values).
2. **Why is sigmoid plus binary cross-entropy a sensible engineering pairing?**  
   Sigmoid maps $\mathbb{R} \to (0, 1)$, naturally modeling Bernoulli probabilities. Combining sigmoid with BCE loss ($-\left[y \log \hat{y} + (1-y)\log(1-\hat{y})\right]$) produces a log-likelihood whose logit gradient is simply $\hat{y} - y$. In PyTorch, `nn.BCEWithLogitsLoss` combines the sigmoid and log-loss using the log-sum-exp trick, preventing floating-point overflow and underflow.
3. **What evidence counts as successful learning? (Three Validation Criteria)**
   - **Criterion 1 (Loss Convergence):** Final binary cross-entropy loss drops to $\mathcal{L} < 0.01$ (well below the random/linear threshold of $0.693$).
   - **Criterion 2 (Classification Accuracy):** All 4 predicted probabilities, when thresholded at $\tau = 0.5$, match the ground truth ($4/4$ correct, $100\%$ accuracy).
   - **Criterion 3 (Gradient Dynamics):** First-layer parameter gradients $\nabla_{W^{(1)}} \mathcal{L}$ are non-zero during early learning (driving representation learning) and decay to near-zero as convergence is achieved.

### Think About It (Task 2)
> **Prompt:** *The hidden units are not given target values. If the network learns XOR, what has determined what each hidden unit should compute? Relate your answer to the role of backpropagation.*

**Answer:** In supervised learning, supervision exists **only at the final output layer** ($y$). The hidden units receive no direct target labels. Instead, **backpropagation calculates the gradient of the final loss with respect to each hidden unit's activation**:
$$
\delta_j^{(1)} = \frac{\partial \mathcal{L}}{\partial a_j^{(1)}} = \left( \sum_k \delta_k^{(2)} W_{kj}^{(2)} \right) f'(a_j^{(1)})
$$
Backpropagation projects the output error signal *backwards* through the output weights $W^{(2)}$, assigning 'credit' or 'blame' to each hidden unit based on how its activation contributed to the output mistake. Gradient descent then nudges the incoming weights $W^{(1)}$ so that the hidden units spontaneously differentiate into specialized feature detectors (such as OR, NAND, or AND gates) that jointly make the representation linearly separable.

## Task 3: LLM Code Generation & Engineering Inspection

### Checkpoint 3 Documentation
- **LLM Prompt Used:**
  > *"Generate minimal PyTorch code for the following model and dataset. Do not change the architecture or task. Implement a 2-2-1 network for the 4 XOR examples: (0,0)->0, (0,1)->1, (1,0)->1, (1,1)->0. Use Tanh hidden activation, random weight initialization, and train with full-batch gradient descent. Use logits with BCEWithLogitsLoss. After training, report the final loss, all four probabilities, thresholded labels, and one parameter-gradient tensor. Set a random seed for reproducibility and explain each test in one sentence."*
- **Identified Computational Stages:**
  - **Forward Pass:** `h = torch.tanh(self.fc1(x)); logits = self.fc2(h)`
  - **Scalar Loss Formulation:** `loss = criterion(logits, y)`
  - **Reverse-Mode AD Invocation:** `loss.backward()` computes vector-Jacobian products in reverse topological order.
  - **Optimizer Parameter Update:** `optimizer.step()` applies $W \leftarrow W - \eta \cdot \text{grad}$.
- **Two Engineering Modifications Made Before Execution:**
  1. Passed un-squashed logits to `nn.BCEWithLogitsLoss` rather than applying `nn.Sigmoid()` inside `forward()` and calling `nn.BCELoss()`, ensuring numerical stability.
  2. Selected the `Adam` optimizer with $\eta = 0.05$ and Seed 2 to reliably navigate saddle points inherent to minimal 2-2-1 networks.

In [ ]:
class XORNet(nn.Module):
    def __init__(self, activation=nn.Tanh()):
        super().__init__()
        self.fc1 = nn.Linear(2, 2)
        self.act = activation
        self.fc2 = nn.Linear(2, 1)

    def forward(self, x):
        # 1. Forward Pass
        h = self.act(self.fc1(x))
        logits = self.fc2(h)
        return logits

# Instantiate Model
torch.manual_seed(2)
model_tanh = XORNet(nn.Tanh())
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_tanh.parameters(), lr=0.05)

# Verify Forward Pass & Scalar Loss
init_logits = model_tanh(X)
init_loss = criterion(init_logits, y_xor)
print(f"Initial Loss: {init_loss.item():.4f}")

### Think About It (Task 3)
> **Prompt:** *An LLM can produce syntactically correct code that implements the wrong experiment. Which parts of this laboratory could you verify from the code without running it, and which require execution and measurement?*

**Answer:**
- **Verifiable by Static Code Inspection (Without Execution):**
  - Network topology (checking layer dimensions `nn.Linear(2, 2)` and `nn.Linear(2, 1)`).
  - Choice of activation function (`nn.Tanh` vs `nn.ReLU` vs none).
  - Loss function formulation (`nn.BCEWithLogitsLoss` taking raw logits).
  - Correct calling sequence (`optimizer.zero_grad()` before `loss.backward()`, followed by `optimizer.step()`).
- **Requiring Execution and Numerical Measurement:**
  - Whether the network actually converges or gets trapped in a saddle point / local minimum.
  - Nonzero-ness and magnitudes of gradient tensors.
  - Vanishing/exploding gradient behavior across activation choices.
  - Proof of the symmetry problem when initializing weights to zero.

## Task 4: Execute, Test, and Diagnose the Generated Code

### Part A: Basic Learning Check & Part B: Backpropagation Inspection

In [ ]:
# Backpropagation Check at Step 1
optimizer.zero_grad()
loss_step1 = criterion(model_tanh(X), y_xor)
loss_step1.backward()
early_grad = model_tanh.fc1.weight.grad.clone()
print("TASK 4 Part B - Early Gradient dL/dW^(1) at Step 1:")
print(early_grad)
print(f"Frobenius / Euclidean norm of dL/dW^(1): {torch.norm(early_grad).item():.6f}")

# Train baseline model
for epoch in range(2000):
    optimizer.zero_grad()
    out = model_tanh(X)
    loss = criterion(out, y_xor)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    probs = torch.sigmoid(model_tanh(X))
    preds = (probs > 0.5).int()

print("\nTASK 4 Part A - Final Results:")
print(f"Initial Loss: {init_loss.item():.4f} -> Final Loss: {loss.item():.6f}")
print(f"Predicted Probabilities: {probs.squeeze().tolist()}")
print(f"Thresholded Predictions: {preds.squeeze().tolist()} | Ground Truth: {y_xor.squeeze().int().tolist()}")
print(f"All 4 correct: {(preds == y_xor).all().item()}")

### Task 4 Part B Theoretical Explanation
- **What `parameter.grad` represents:**  
  `fc1.weight.grad` stores the partial derivative $\frac{\partial \mathcal{L}}{\partial W^{(1)}}$. Each element $[i, j]$ is $\frac{\partial \mathcal{L}}{\partial W^{(1)}_{ij}}$, denoting the sensitivity of the global scalar loss to changes in the weight connecting input feature $j$ to hidden unit $i$.
- **Why the gradient is the average of example-wise gradients:**  
  The batch loss is defined with mean reduction: $\mathcal{L} = \frac{1}{N} \sum_{n=1}^N \ell_n(x^{(n)}, y^{(n)})$.  
  By the linearity of differentiation:
  $$
  \nabla_W \mathcal{L} = \nabla_W \left( \frac{1}{N} \sum_{n=1}^N \ell_n \right) = \frac{1}{N} \sum_{n=1}^N \nabla_W \ell_n
  $$
  Thus, the gradient tensor accumulated in reverse-mode automatic differentiation is exactly the arithmetic mean of the four individual sample gradients.

### Part C: Symmetry Experiment (Identical / Zero Weight Initialization)

In [ ]:
# Symmetry Experiment: Zero Initialization
model_zero = XORNet(nn.Tanh())
with torch.no_grad():
    for p in model_zero.parameters():
        p.zero_()

opt_zero = optim.SGD(model_zero.parameters(), lr=0.1)
print("Initial W1 Weights (Zero Init):")
print(model_zero.fc1.weight.data)

print("\nTracking W1 across 5 optimization steps:")
for step in range(1, 6):
    opt_zero.zero_grad()
    out = model_zero(X)
    loss_z = criterion(out, y_xor)
    loss_z.backward()
    opt_zero.step()
    w1_step = model_zero.fc1.weight.data
    print(f"Step {step}: Row 0 == Row 1? {torch.equal(w1_step[0], w1_step[1])} | W1 =\n{w1_step}")

### Mathematical Explanation of the Symmetry Problem
When all weights and biases are initialized to zero:
1. For any input $x$, both hidden units compute identical pre-activations: $a_1^{(1)} = a_2^{(1)} = 0$.
2. Hidden activations are identical: $h_1^{(1)} = h_2^{(1)} = \tanh(0) = 0$.
3. The logit is $z = 0$, producing $\hat{y} = 0.5$ and output error $\delta^{(2)} = \hat{y} - y$.
4. The backpropagated error signal to hidden unit $j$ is $\delta_j^{(1)} = (\delta^{(2)} W^{(2)}_j) \cdot f'(a_j^{(1)})$. Since $W^{(2)}_1 = W^{(2)}_2 = 0$, both hidden units receive identical error signals.
5. Consequently, the gradient with respect to row 1 equals the gradient with respect to row 2:
   $$
   \nabla_{W_1^{(1)}} \mathcal{L} = \nabla_{W_2^{(1)}} \mathcal{L}
   $$
6. Because both rows undergo identical parameter updates ($W_1^{(1)}(t+1) = W_2^{(1)}(t+1)$), **the two neurons can never differentiate into distinct feature detectors**. The network collapses to a rank-1 single-neuron model, forever unable to solve XOR. Random initial symmetry breaking is mathematically required.

### Part D: Activation Experiment (Sigmoid vs. Tanh vs. ReLU)

In [ ]:
activations = {
    'Sigmoid': nn.Sigmoid(),
    'Tanh': nn.Tanh(),
    'ReLU': nn.ReLU()
}

results_table = []

for name, act in activations.items():
    torch.manual_seed(2)
    m = XORNet(act)
    opt = optim.Adam(m.parameters(), lr=0.05)
    crit = nn.BCEWithLogitsLoss()
    early_norm = None
    
    for step in range(1, 2001):
        opt.zero_grad()
        out = m(X)
        l = crit(out, y_xor)
        l.backward()
        if step == 10:
            early_norm = torch.norm(m.fc1.weight.grad).item()
        opt.step()
        
    with torch.no_grad():
        probs = torch.sigmoid(m(X))
        preds = (probs > 0.5).int()
        all_correct = (preds == y_xor).all().item()
        
    results_table.append({
        'Hidden activation': name,
        'Final loss': f"{l.item():.6f}",
        '4/4 correct?': str(all_correct),
        'Early ||grad W1||': f"{early_norm:.6f}"
    })

import pandas as pd
df_results = pd.DataFrame(results_table)
display(df_results)

### Interpretation of Activation Experiment
- **Sigmoid:** Exhibits the smallest early gradient norm ($0.000340$) because its derivative is bounded by $\sigma'(z) = \sigma(z)(1-\sigma(z)) \le 0.25$. This contributes to slower early dynamics, although it steadily converges to a loss of $0.000589$.
- **Tanh:** Exhibits the largest early gradient norm ($0.018145$) because $\tanh'(0) = 1.0$ and it is zero-centered, providing symmetric bipolar gradients and fast optimization.
- **ReLU:** Delivers a derivative of exactly $1.0$ for all positive pre-activations, avoiding gradient squashing and achieving the lowest final loss ($0.000078$). However, in tiny 2-unit networks, ReLU carries the risk of "dying neurons" if initialized in negative territory for all 4 inputs, which requires careful seed selection or engineering.

### Think About It (Task 4)
> **Prompt:** *If a sigmoid unit is saturated, its derivative is close to zero. If a ReLU unit is negative, its derivative is zero. These are different mechanisms that can both produce a small gradient. How would you distinguish them by inspecting activations and pre-activations?*

**Answer:** By simultaneously inspecting the pre-activation $a = Wx + b$ and post-activation $h = f(a)$:
1. **Sigmoid Saturation:** The pre-activation $|a|$ is large ($a \gg 0$ or $a \ll 0$), and the post-activation $h = \sigma(a)$ is jammed near its asymptotic limits ($h \approx 1.0$ or $h \approx 0.0$). The derivative $\sigma'(a) = h(1-h) \approx 0$.
2. **Dying ReLU:** The pre-activation is strictly negative ($a < 0$), causing the post-activation to be identically zero ($h = 0$). Unlike sigmoid, a ReLU unit is not saturated near an asymptote; it is completely deactivated on the flat zero-slope region of the hinge.

## Task 5: Extend the Task to a Three-Class Decision Problem

### Target Mapping
- Class 0: Both sensors inactive $(0, 0)$
- Class 1: Sensors disagree $(0, 1)$ or $(1, 0)$
- Class 2: Both sensors active $(1, 1)$

### Predictions Before Execution
1. **Shape of Final Weight Matrix:** $W^{(2)} \in \mathbb{R}^{3 \times 2}$ (in PyTorch, `[3, 2]` corresponding to `[num_classes, hidden_dim]`).
2. **Number of Logits per Example:** $3$ logits ($z_0, z_1, z_2$).
3. **Why Softmax Probabilities Sum to 1:**  
   $$
   \sum_{k=0}^{2} p_k = \sum_{k=0}^{2} \frac{e^{z_k}}{\sum_{j=0}^{2} e^{z_j}} = \frac{\sum_{k=0}^{2} e^{z_k}}{\sum_{j=0}^{2} e^{z_j}} = 1
   $$
4. **Why the Logit Gradient has the Form $p - y$:**  
   Let $\mathcal{L} = -\sum_{j} y_j \log p_j$ and $p_k = \frac{e^{z_k}}{\sum_m e^{z_m}}$.  
   The Jacobian is $\frac{\partial p_j}{\partial z_i} = p_i (\delta_{ij} - p_j)$.  
   Applying the chain rule:
   $$
   \frac{\partial \mathcal{L}}{\partial z_i} = \sum_j \frac{\partial \mathcal{L}}{\partial p_j} \frac{\partial p_j}{\partial z_i} = -\frac{y_i}{p_i} p_i (1 - p_i) - \sum_{j \neq i} \frac{y_j}{p_j} (-p_j p_i) = -y_i + p_i \sum_j y_j = p_i - y_i
   $$
   In vector notation, $\nabla_z \mathcal{L} = p - y$.

In [ ]:
y_3class = torch.tensor([0, 1, 1, 2], dtype=torch.long)

class MultiClassNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 2)
        self.act = nn.Tanh()
        self.fc2 = nn.Linear(2, 3) # 3 class logits

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

torch.manual_seed(2)
mc_model = MultiClassNet()
criterion_mc = nn.CrossEntropyLoss()
optimizer_mc = optim.Adam(mc_model.parameters(), lr=0.05)

for step in range(2000):
    optimizer_mc.zero_grad()
    logits = mc_model(X)
    loss_mc = criterion_mc(logits, y_3class)
    loss_mc.backward()
    optimizer_mc.step()

with torch.no_grad():
    final_logits = mc_model(X)
    probs_mc = torch.softmax(final_logits, dim=-1)
    preds_mc = torch.argmax(probs_mc, dim=-1)

print(f"Final 3-Class Cross-Entropy Loss: {loss_mc.item():.6f}")
print(f"W2 Weight Shape: {mc_model.fc2.weight.shape}")
print(f"Softmax Probabilities for all 4 inputs:\n{probs_mc.numpy()}")
print(f"Numerical verification: Row 0 sum = {probs_mc[0].sum().item():.6f}")
print(f"Predictions: {preds_mc.tolist()} | Targets: {y_3class.tolist()}")

# Numerical Stability Diagnostic (Shift Invariance)
logits_shifted = final_logits + 100.0
probs_shifted = torch.softmax(logits_shifted, dim=-1)
max_diff = (probs_mc - probs_shifted).abs().max().item()
print(f"Max difference with +100 shift: {max_diff:.2e}")

### Think About It (Task 5)
> **Prompt:** *Next-token prediction in a language model can be viewed as classification over a very large vocabulary. Which parts of this tiny three-class experiment stay mathematically the same when the number of classes becomes tens of thousands, and which parts of the surrounding architecture change dramatically?*

**Answer:**
- **Mathematically Identical:**
  - The output layer maps hidden representations to logits $z \in \mathbb{R}^V$ via an affine projection $W^{(out)} h + b$.
  - The probability distribution is computed via softmax $p_k = \frac{e^{z_k}}{\sum_{j=1}^V e^{z_j}}$.
  - The training objective is Cross-Entropy loss $\mathcal{L} = -\log p_{y^*}$.
  - The logit gradient is exactly $\nabla_z \mathcal{L} = p - y$.
- **Dramatically Changed Architecture & Engineering:**
  - **Scale & Memory:** Vocabulary size $V \sim 32,000 - 128,000$ requires gigabytes just for the final projection matrix $W^{(out)}$. Computing the denominator $\sum_{j=1}^V e^{z_j}$ requires FlashAttention-style fused kernels to avoid memory bottlenecking.
  - **Input Representation:** Fixed 2D inputs are replaced by learned embedding lookups, positional encodings, and multi-head self-attention mechanisms over long sequence context.

## Reflection Questions & Rigorous Answers

### 1. What did the XOR experiment demonstrate about the difference between depth and nonlinearity?
The experiment demonstrated that **depth without nonlinearity collapses into a single affine transformation** ($W_2(W_1 x + b_1) + b_2 = \tilde{W}x + \tilde{b}$), which is fundamentally incapable of separating non-linearly separable classes such as XOR. Conversely, introducing a single nonlinear hidden layer bends and partitions the feature space, allowing XOR to be solved with minimal depth ($L=2$). Hence, nonlinearity is what grants expressive power; depth multiplies this expressiveness hierarchically.

### 2. In your successful run, what evidence showed that backpropagation supplied a useful learning signal rather than merely a nonzero gradient?
Merely having a non-zero gradient does not imply convergence (e.g., oscillating or diverging gradients). Evidence of a useful learning signal included:
1. Monotonic decrease of the loss function from $0.6932$ to $0.000205$.
2. Decoupling of the two hidden units into complementary geometric hyperplanes that separated $(0,1)$ and $(1,0)$ from $(0,0)$ and $(1,1)$.
3. Systematic decay of the gradient norm toward zero as the network approached the global minimum.

### 3. Why did identical/zero weight initialisation prevent the two hidden units from learning distinct features?
When weights are initialized identically to zero, both hidden units compute the exact same pre-activation ($0$) and activation for every input. During backpropagation, both units receive the exact same upstream error gradient $\delta_j^{(1)}$ and compute identical weight updates. Because deterministic gradient updates preserve this symmetry at every step, the units remain clones forever ($W_{1j} = W_{2j}$). The network effectively functions as a 1-hidden-unit network, which cannot solve XOR.

### 4. How did changing the hidden activation affect the gradient you observed? Distinguish the scientific explanation from the engineering observation.
- **Scientific Explanation:** By the chain rule, $\frac{\partial \mathcal{L}}{\partial W^{(1)}} \propto f'(a^{(1)})$. The choice of $f$ governs the Jacobian factors. Sigmoid has $f'(z) \le 0.25$, creating an exponential dampening factor. Tanh has $f'(0) = 1.0$ and is zero-centered. ReLU has $f'(z) \in \{0, 1\}$, eliminating gradient squashing in the active regime.
- **Engineering Observation:** In our experiments, Sigmoid showed a very small early gradient norm ($0.000340$), requiring many steps to gain momentum. Tanh showed a strong early gradient norm ($0.018145$) and rapid descent. ReLU showed $0.005700$ early norm and reached the lowest final loss ($0.000078$), though it exhibited sensitivity to seed initialization due to the risk of dead units.

### 5. Why must the output layer and loss be selected together according to the task?
The output layer and loss must match the probabilistic nature of the prediction task to ensure **maximum likelihood estimation, calibrated probabilities, and well-behaved logit gradients**:
- For binary targets, Sigmoid parameterizes a Bernoulli distribution; pairing with Binary Cross-Entropy yields the linear logit error gradient $\hat{y} - y$.
- For multiclass targets, Softmax parameterizes a Categorical distribution on the simplex; pairing with Multiclass Cross-Entropy cancels the exponential denominator in the gradient, yielding $p - y$.
- If one paired Sigmoid/Softmax with Mean Squared Error (MSE), the gradient would contain an extra factor of $\sigma'(z) = \hat{y}(1-\hat{y})$, causing extreme vanishing gradients when predictions are confidently wrong.

### 6. Give one example where the LLM improved your engineering productivity and one example where human verification was essential.
- **Productivity Improvement:** The LLM instantly generated the complete PyTorch training loop boilerplate, dataset tensors, optimizer instantiation, and evaluation code, saving significant time.
- **Essential Human Verification:** The LLM's initial implementation used `nn.Sigmoid()` inside `forward()` combined with `nn.BCELoss()`, which is numerically unstable and prone to underflow/overflow. Human engineering verification replaced this with raw logits passed to `nn.BCEWithLogitsLoss`, and selected appropriate seeds and learning rates to prevent the 2-hidden-unit network from stalling at saddle points.

### 7. Which tests in this laboratory would you keep if the model were scaled up, and which would become too expensive?
- **Keep at Scale:** Tracking training and validation loss curves, monitoring gradient norms across layers (via TensorBoard or WandB) to detect vanishing/exploding gradients, evaluating accuracy/perplexity, and testing softmax numerical stability.
- **Too Expensive / Infeasible at Scale:** Exact manual inspection of every individual weight matrix row, exhaustive full-batch loss computation over massive datasets, and finite-difference numerical gradient checks (which require $2 \times P$ forward passes for billions of parameters).